# 🧠 AI Logic Engine: Knowledge Ingestion (Extreme Debug & Speed Pass)

### 🚀 Update Log (v4.3):
- **Target:** General Knowledge, Greetings, Manners, Social Phrases.
- **Real-time Debug:** AI ဘာတွေဖြေနေလဲ၊ Database မှာ ဘာတွေသိမ်းနေလဲဆိုတာကို Step-by-step ထုတ်ပြပေးပါမယ်။
- **Instruction:** Runtime ကို `T4 GPU` ထားရန် အထူးသတိပြုပါ။

In [ ]:
# @title 🛠️ Step 1: Install & Imports
import time
print("⏳ Installing libraries...")
!pip install -q firebase-admin transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, datetime
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Setup Complete.")

In [ ]:
# @title 🔑 Step 2: Firestore Connection
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Service Account Key (JSON) ကို Upload တင်ပေးပါ...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("❌ JSON key လိုအပ်ပါသည်။")
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database=DATABASE_ID)
print(f"✅ Firestore Synced: {DATABASE_ID}")

In [ ]:
# @title 🧬 Step 3: Logic Engine Config
STATE_FILE = "ingestion_state_v4_3.json"

def clean_id(text):
    if not text: return ""
    text = str(text).lower().strip()
    # Support Unicode but replace forbidden characters for Firestore IDs
    return re.sub(r'[\s/\\.#\[\]\*\?\!]+', '_', text).strip('_')

def normalize_verb(v):
    v = str(v).strip().lower()
    is_a_set = ['is_a', 'is a', 'is type of', 'belongs to', 'category of', 'member of']
    if any(p == v for p in is_a_set): return 'is_a'
    return v.replace(' ', '_')

def save_state(count):
    with open(STATE_FILE, "w") as f: json.dump({"count": count, "timestamp": str(datetime.datetime.now())}, f)

def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f: return json.load(f).get("count", 0)
        except: pass
    return 0

print("✅ Logic Configured.")

In [ ]:
# @title 🤖 Step 4: Load Faster Model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"⏳ Loading {model_id} (Quantized 4-bit)... ")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)
model.eval()
print(f"✅ Model Loaded on: {model.device}")

In [ ]:
# @title 🚀 Step 5: Start Continuous Ingestion
TARGET_FACT_COUNT = 1000000 # @param {type:"number"}
CATEGORIES = [
    "General Knowledge", "Small Talk Greetings", "Social Manners", 
    "Daily Etiquette", "Common Jokes", "Basic Landmarks", 
    "Meeting New People", "Office Manners", "Travel Tips"
]

current_count = load_state()
pbar = tqdm(initial=current_count, total=TARGET_FACT_COUNT, desc="Syncing Symbols")

print(f"\n🔥 INGESTION STARTING... (Target Goal: {TARGET_FACT_COUNT})")
print("Logs will show every AI generation to ensure proof of work.\n")

while current_count < TARGET_FACT_COUNT:
    loop_start = time.time()
    try:
        cat = CATEGORIES[current_count % len(CATEGORIES)]
        
        # Faster System Prompt
        prompt = f"<|im_start|>system\nYou are a precise knowledge extractor. Output facts in format: Subject|Verb|Object|FullSentence. No intro, no chat.<|im_end|>\n<|im_start|>user\nList 20 distinct facts about {cat}.<|im_end|>\n<|im_start|>assistant\n"
        
        # 1. Tokenizing
        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
        
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] ⏳ AI generating facts for: {cat}...", end=" ")
        
        # 2. Generating
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids, 
                attention_mask=inputs.attention_mask, 
                max_new_tokens=512, 
                do_sample=True, 
                temperature=0.7,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True
            )
        
        # 3. Decoding
        response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        if not response: 
            print("❌ AI returned empty response. Retrying...")
            continue
            
        print("✅ Received.")
        
        # 4. Parsing & DB Write
        batch = db.batch()
        added_this_loop = 0
        lines = response.split('\n')
        
        for line in lines:
            if '|' not in line: continue
            parts = [p.strip() for p in line.split('|')]
            
            if len(parts) >= 3:
                s, v, o = parts[0], normalize_verb(parts[1]), parts[2]
                sid, oid = clean_id(s), clean_id(o)
                
                if sid and oid and sid != oid:
                    node_ref = db.collection('nodes').document(sid)
                    data = {'name': s, 'updatedAt': firestore.SERVER_TIMESTAMP}
                    
                    if v == 'is_a':
                        data['groups'] = firestore.ArrayUnion([oid])
                    else:
                        sent = parts[3] if len(parts) > 3 else f"{s} {v.replace('_',' ')} {o}."
                        data['relations'] = firestore.ArrayUnion([{
                            'verb': v, 'targetId': oid, 'sentence': sent, 'weight': 100
                        }])
                    
                    batch.set(node_ref, data, merge=True)
                    added_this_loop += 1
        
        # 5. Commit Check
        if added_this_loop > 0:
            print(f"   ☁️ Syncing {added_this_loop} symbols to Firestore...", end=" ")
            batch.commit()
            current_count += added_this_loop
            pbar.update(added_this_loop)
            save_state(current_count)
            print("Done.")
        else:
            print(f"   ⚠️ No valid triplets found in: {response[:50]}...")
            
        # 6. Memory Maintenance
        if current_count % 30 == 0: torch.cuda.empty_cache()
        
        # Speed check
        elapsed = time.time() - loop_start
        # print(f"   ⏱️ Loop Time: {elapsed:.2f}s")
        
    except Exception as e:
        print(f"\n❌ [SYSTEM ERROR] {str(e)}")
        print("⏳ Waiting 5s before recovery...\n")
        time.sleep(5)
    except KeyboardInterrupt:
        print("\n🛑 Manually Stopped. Finalizing state...")
        break

pbar.close()
print(f"✅ Finished. Total Concepts in Database: {current_count}")